# FoodSeg103 Rebalanced Audit

Notebook này chạy full audit cho `source/dataset/foodseg103_rebalanced` và ghi output vào `source/dataset/foodseg103_rebalanced/docs`.

In [1]:
from pathlib import Path
import sys

# Ensure project root is importable when notebook runs from source/preprocessing
project_root = Path.cwd()
if (project_root / "script_audit.py").exists():
    root = project_root
else:
    root = project_root.parent.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Using project root: {root}")

Using project root: F:\ANHTHU\1-HCMUS\1 - STUDY\HKVIII\CV\PROJECT


In [2]:
from script_audit import main

main()
print("Audit done. Check source/dataset/foodseg103_rebalanced/docs")

train ann=3981, train masks=3981
test ann=2135, test masks=2135
Audit completed. Outputs saved to docs/ and docs/patches/
Audit done. Check source/dataset/foodseg103_rebalanced/docs


Audit completed. Outputs saved to docs/ and docs/patches/
Audit done. Check source/dataset/foodseg103_rebalanced/docs


In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from script_audit import DATA_DIR, resolve_image_path

SEAWEED_ID = 65
MAX_EXAMPLES = 5


def find_masks_with_label(label_id: int, max_examples: int = 5) -> list[dict]:
    """Find mask files that contain a target label id.

    Args:
        label_id: Target class id to search inside masks.
        max_examples: Maximum number of examples to return.
    Returns:
        list[dict]: Example metadata including stem, mask path, unique ids, and label pixels.
    Raises:
        RuntimeError: If a mask file cannot be decoded.
    """
    results = []
    for split in ["train", "test"]:
        mask_dir = DATA_DIR / split / "mask"
        for mask_path in sorted(mask_dir.glob("*.png")):
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                raise RuntimeError(f"Cannot read mask: {mask_path}")
            label_pixels = int(np.count_nonzero(mask == label_id))
            if label_pixels == 0:
                continue
            unique_ids = [int(x) for x in np.unique(mask)]
            results.append(
                {
                    "split": split,
                    "stem": mask_path.stem,
                    "mask_path": str(mask_path),
                    "unique_label_ids": unique_ids,
                    "label_65_pixels": label_pixels,
                }
            )
            if len(results) >= max_examples:
                return results
    return results



def build_overlay(rgb_image: np.ndarray, binary_mask: np.ndarray) -> np.ndarray:
    """Build an RGB overlay for visual inspection.

    Args:
        rgb_image: Source RGB image.
        binary_mask: Binary mask with foreground as 1.
    Returns:
        np.ndarray: RGB overlay image.
    Raises:
        ValueError: If image and mask sizes do not match.
    """
    if rgb_image.shape[:2] != binary_mask.shape[:2]:
        raise ValueError("Image and mask must have the same spatial size.")
    overlay = rgb_image.copy()
    red_layer = np.zeros_like(rgb_image)
    red_layer[..., 0] = 255
    mask_bool = binary_mask.astype(bool)
    overlay[mask_bool] = (0.65 * overlay[mask_bool] + 0.35 * red_layer[mask_bool]).astype(np.uint8)
    return overlay



def show_seaweed_examples(records: list[dict], label_id: int) -> None:
    """Display RGB, binary mask, and overlay for each record.

    Args:
        records: Records returned by `find_masks_with_label`.
        label_id: Target class id to visualize.
    Returns:
        None.
    Raises:
        RuntimeError: If any source image cannot be decoded.
    """
    if not records:
        print(f"No mask contains label {label_id}.")
        return

    for record in records:
        mask_path = Path(record["mask_path"])
        image_path = resolve_image_path(DATA_DIR / record["split"] / "img", record["stem"])
        if image_path is None:
            raise RuntimeError(f"Cannot resolve image for {record['stem']}")

        bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if bgr is None:
            raise RuntimeError(f"Cannot read image: {image_path}")
        if mask is None:
            raise RuntimeError(f"Cannot read mask: {mask_path}")

        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        binary_mask = (mask == label_id).astype(np.uint8)
        overlay = build_overlay(rgb, binary_mask)

        print(f"stem: {record['stem']}")
        print(f"mask_path: {record['mask_path']}")
        print(f"unique_label_ids: {record['unique_label_ids']}")
        print(f"pixels_of_{label_id}: {record['label_65_pixels']}")

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(rgb)
        axes[0].set_title(f"RGB | {record['split']} | {record['stem']}")
        axes[1].imshow(binary_mask, cmap="gray")
        axes[1].set_title(f"Mask label {label_id}")
        axes[2].imshow(overlay)
        axes[2].set_title("Overlay")
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()


seaweed_examples = find_masks_with_label(label_id=SEAWEED_ID, max_examples=MAX_EXAMPLES)
print(f"Found {len(seaweed_examples)} mask(s) containing label {SEAWEED_ID}.")
display(pd.DataFrame(seaweed_examples))
show_seaweed_examples(seaweed_examples, label_id=SEAWEED_ID)